# Notebook 6: Naïve Bayes Classifier
## Gaussian NB, Multinomial NB, and Probabilistic Reasoning

**Difficulty:** ⭐⭐⭐ &nbsp;|&nbsp; **Estimated Time:** 2.5 hours &nbsp;|&nbsp; **Week 6 — Classification**

---

## Why This Matters

Naïve Bayes is famous for one thing: **spam filtering is where it got its reputation.** In the late 1990s, Paul Graham showed that a simple probabilistic model based on word frequencies could outperform far more complex hand-crafted rules. Gmail, Yahoo Mail, and virtually every email provider still uses Bayes-family techniques as a first-pass filter.

Naïve Bayes is also the canonical example of a **generative model** — it learns how each class *generates* data, rather than just learning a boundary between classes. This distinction matters:
- **Discriminative models** (logistic regression, SVM) ask: *"Given x, what is P(y|x)?"*
- **Generative models** (Naïve Bayes) ask: *"How does class y generate data x? What is P(x|y)?"*

---

## Real-World Analogy First

Imagine you are a postal inspector trying to decide whether a letter is spam or legitimate. You have two drawers of reference letters:
- **Drawer A (Spam):** Contains thousands of known spam letters
- **Drawer B (Legit):** Contains thousands of known legitimate letters

A new letter arrives. Your process:
1. **Base rate check:** *"How common is spam overall? About 40% of all letters."* This is your **prior** — your belief before reading a single word.
2. **Evidence check:** *"This letter contains the word 'WINNER'. How often does 'WINNER' appear in Drawer A vs Drawer B?"* This is the **likelihood** — how probable the evidence is under each hypothesis.
3. **Update your belief:** Combine prior × likelihood → **posterior probability** of spam.

The **"naïve"** part: you assume each word's evidence is **independent** of every other word. In reality, "cheap" and "pills" co-occur together in spam — they are correlated. Naïve Bayes ignores this correlation. Yet despite this blatant approximation, it works remarkably well in practice.

---

## Learning Objectives

By the end of this notebook you will be able to:
1. Apply Bayes' theorem to email classification
2. Explain the naive independence assumption and why it works despite being wrong
3. Implement Gaussian NB from scratch (fit + predict)
4. Use Multinomial NB for word-count features with Laplace smoothing
5. Demonstrate the zero-probability problem and why Laplace smoothing fixes it
6. Sample synthetic spam using the generative model
7. Compare NB with Logistic Regression on accuracy, speed, and calibration

## Section 1: Bayes' Theorem for Email Classification

### The Formula

$$P(\text{spam} \mid \text{words}) = \frac{P(\text{words} \mid \text{spam}) \cdot P(\text{spam})}{P(\text{words})}$$

Each term has a name and a role:

| Term | Symbol | Meaning | How to compute |
|------|--------|---------|----------------|
| **Prior** | P(spam) | Base rate of spam in your inbox | Count: fraction of training emails labelled spam |
| **Likelihood** | P(words\|spam) | How probable these words are *given* spam | Learn from spam examples |
| **Evidence** | P(words) | How probable these words are overall | Constant — same for all classes |
| **Posterior** | P(spam\|words) | Updated belief after seeing the email | Computed via Bayes' theorem |

### The Naïve Independence Assumption

Computing P(all words | spam) jointly requires knowing how every word co-occurs with every other — an exponential number of parameters. Instead, Naïve Bayes assumes:

$$P(w_1, w_2, \ldots, w_n \mid \text{spam}) = P(w_1 \mid \text{spam}) \cdot P(w_2 \mid \text{spam}) \cdots P(w_n \mid \text{spam})$$

Each word is treated as **conditionally independent** given the class. This is almost never true — *"free offer"* has very different meaning from *"free"* and *"offer"* separately. But the independence assumption slashes the parameter count from exponential to linear, and the resulting classifier is often competitive with far more complex models.

### Why the Denominator Doesn't Matter for Classification

When comparing P(spam|words) vs P(legitimate|words), P(words) is identical for both — it cancels out. We only need:

$$\hat{y} = \arg\max_k \; P(y = k) \cdot \prod_j P(x_j \mid y = k)$$

In practice we work in **log-space** to avoid floating-point underflow from multiplying many small probabilities:

$$\hat{y} = \arg\max_k \left[ \log P(y = k) + \sum_j \log P(x_j \mid y = k) \right]$$

In [ ]:
# ── Imports and reproducibility ────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import warnings
from scipy.stats import norm

from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
np.random.seed(42)

# Colour scheme: binary classification (spam=red, not-spam=blue)
# Extended for 4-class later
SPAM_COLOR  = '#e74c3c'   # red
LEGIT_COLOR = '#3498db'   # blue

CLASS_NAMES_4  = ['Spam', 'Promotional', 'Personal', 'Work']
CLASS_COLORS_4 = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

print('Libraries loaded.')
print('sklearn Naive Bayes variants available: GaussianNB, MultinomialNB, BernoulliNB')

## Section 2: Bayes' Theorem — Manual Walkthrough

Before any code, let's compute Bayes' theorem by hand on a tiny example. This builds the intuition we'll then scale up.

In [ ]:
# ── Manual Bayes' theorem walkthrough ─────────────────────────────────────────
print('=== Bayes\' Theorem: Manual Email Classification ===')
print()

# Step 1: Prior probabilities (base rates in inbox)
p_spam  = 0.40   # 40% of all emails are spam
p_legit = 0.60   # 60% are legitimate
print(f'Step 1 — Prior probabilities:')
print(f'  P(spam)       = {p_spam:.2f}')
print(f'  P(legitimate) = {p_legit:.2f}')

# Step 2: Likelihood — P(word | class), estimated from training data
# Suppose we have two words: "WINNER" and "Dear"
p_winner_given_spam  = 0.30   # "WINNER" appears in 30% of spam
p_winner_given_legit = 0.01   # "WINNER" appears in  1% of legit
p_dear_given_spam    = 0.05   # "Dear" appears in 5% of spam
p_dear_given_legit   = 0.60   # "Dear" appears in 60% of legit

print(f'\nStep 2 — Likelihoods (from training data):')
print(f'  P(WINNER | spam)  = {p_winner_given_spam}')
print(f'  P(WINNER | legit) = {p_winner_given_legit}')
print(f'  P(Dear   | spam)  = {p_dear_given_spam}')
print(f'  P(Dear   | legit) = {p_dear_given_legit}')

# Step 3: Numerator scores (unnormalised posteriors) — assume independence
spam_score  = p_spam  * p_winner_given_spam  * p_dear_given_spam
legit_score = p_legit * p_winner_given_legit * p_dear_given_legit

# Step 4: Normalise to get true posteriors
total = spam_score + legit_score
p_spam_posterior  = spam_score  / total
p_legit_posterior = legit_score / total

print(f'\nNew email contains: "WINNER" + "Dear"')
print(f'\nStep 3 — Unnormalised scores (prior × likelihood product):')
print(f'  spam  : {p_spam} × {p_winner_given_spam} × {p_dear_given_spam} = {spam_score:.6f}')
print(f'  legit : {p_legit} × {p_winner_given_legit} × {p_dear_given_legit} = {legit_score:.6f}')

print(f'\nStep 4 — Posterior probabilities (after normalisation):')
print(f'  P(spam  | WINNER, Dear) = {p_spam_posterior:.4f} ({p_spam_posterior*100:.1f}%)')
print(f'  P(legit | WINNER, Dear) = {p_legit_posterior:.4f} ({p_legit_posterior*100:.1f}%)')
print(f'\nVerdict: {"SPAM" if p_spam_posterior > 0.5 else "LEGITIMATE"}')
print()
print('Intuition: "Dear" pushes toward legit, but "WINNER" is so strongly')
print('spam-associated (30× more likely in spam) that the overall verdict is spam.')

## Section 3: Gaussian Naïve Bayes

### When to Use Gaussian NB

When features are **continuous** (real-valued), we model each feature as a Gaussian (Normal) distribution within each class:

$$P(x_j \mid y = k) = \mathcal{N}(x_j; \mu_{jk}, \sigma^2_{jk})$$

**Learning:** Estimate the mean $\mu_{jk}$ and standard deviation $\sigma_{jk}$ for each feature $j$ and class $k$ from training data.

**Prediction:** Plug in the test point and compute the Gaussian PDF value as the likelihood. Take log + argmax.

There are as many Gaussians as **(features × classes)**. For our 5-feature binary problem: 5 × 2 = 10 Gaussians to learn.

In [ ]:
# ── Build spam/legitimate email dataset (continuous features) ─────────────────
np.random.seed(42)
n = 600   # 300 spam + 300 legitimate

# Feature distributions — deliberately different per class so NB can separate them
spam_features = np.column_stack([
    np.random.normal(350,  50,  n//2),   # email_length: spam is long
    np.random.normal( 12,   3,  n//2),   # num_links: spam has many links
    np.random.binomial(1, 0.10, n//2),   # has_greeting: spam rarely personalised
    np.random.normal(  2,   1,  n//2),   # formal_words: spam is informal
    np.random.normal(  1,   1,  n//2).clip(0), # num_attachments
])
legit_features = np.column_stack([
    np.random.normal(180,  60,  n//2),   # email_length: legit is shorter
    np.random.normal(  2,   2,  n//2),   # num_links: few links
    np.random.binomial(1, 0.70, n//2),   # has_greeting: usually personalised
    np.random.normal(  8,   3,  n//2),   # formal_words: more formal
    np.random.normal(  2,   1,  n//2).clip(0), # num_attachments
])

X_bin = np.vstack([spam_features, legit_features]).clip(0)  # no negatives
y_bin = np.array([0]*(n//2) + [1]*(n//2))   # 0=spam, 1=legitimate

feature_names = ['email_length', 'num_links', 'has_greeting', 'formal_words', 'num_attachments']
df_bin = pd.DataFrame(X_bin, columns=feature_names)
df_bin['label'] = ['Spam' if y == 0 else 'Legitimate' for y in y_bin]

X_tr, X_te, y_tr, y_te = train_test_split(X_bin, y_bin, test_size=0.25,
                                            random_state=42, stratify=y_bin)
print(f'Dataset: {X_bin.shape}  |  Train: {X_tr.shape}  |  Test: {X_te.shape}')
print(f'\nClass means:')
print(df_bin.groupby('label')[feature_names].mean().round(2).to_string())

In [ ]:
# ── Class-conditional Gaussian distributions: 2×2 subplot ─────────────────────
# Plot the PDF of each feature for spam vs legitimate — shows how well NB can separate them
plot_features = feature_names[:4]   # pick first 4 features for a clean 2×2 grid

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Class-Conditional Gaussian Distributions per Feature\n'
             '(Naïve Bayes learns one Gaussian per class per feature)',
             fontsize=13, fontweight='bold')

for ax, feat in zip(axes.flat, plot_features):
    spam_vals  = df_bin[df_bin['label'] == 'Spam'][feat].values
    legit_vals = df_bin[df_bin['label'] == 'Legitimate'][feat].values

    # Compute mean and std for each class (what NB learns)
    mu_spam,  std_spam  = spam_vals.mean(),  spam_vals.std()
    mu_legit, std_legit = legit_vals.mean(), legit_vals.std()

    # Draw Gaussian PDF curves — this is P(feature | class)
    x_plot = np.linspace(min(spam_vals.min(), legit_vals.min()),
                         max(spam_vals.max(), legit_vals.max()), 300)
    ax.plot(x_plot, norm.pdf(x_plot, mu_spam,  std_spam),
            color=SPAM_COLOR,  linewidth=2.5, label=f'Spam: μ={mu_spam:.1f}, σ={std_spam:.1f}')
    ax.plot(x_plot, norm.pdf(x_plot, mu_legit, std_legit),
            color=LEGIT_COLOR, linewidth=2.5, label=f'Legit: μ={mu_legit:.1f}, σ={std_legit:.1f}')

    # Shade the overlap region (classification difficulty zone)
    ax.fill_between(x_plot,
                    np.minimum(norm.pdf(x_plot, mu_spam, std_spam),
                               norm.pdf(x_plot, mu_legit, std_legit)),
                    alpha=0.25, color='purple', label='Overlap (ambiguous zone)')

    ax.set_title(feat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature value'); ax.set_ylabel('P(feature | class)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print('Purple shading = overlap region where both classes assign similar likelihood.')
print('Wider separation = more discriminative feature.')

## Section 4: Gaussian NB — From Scratch Implementation

We implement the full Gaussian NB classifier step by step. The math is surprisingly simple:

**Fit:** For each class k and feature j, compute $\mu_{jk} = \text{mean}(x_j \mid y=k)$ and $\sigma_{jk} = \text{std}(x_j \mid y=k)$.

**Predict:** For a test point x, compute:
$$\log\text{score}(k) = \log P(y=k) + \sum_j \log \mathcal{N}(x_j; \mu_{jk}, \sigma_{jk})$$

Return the class k with the highest log-score.

In [ ]:
# ── Gaussian NB from scratch ───────────────────────────────────────────────────
class GaussianNBScratch:
    """Gaussian Naïve Bayes implemented from first principles."""

    def fit(self, X, y):
        """
        Learn class priors and per-class, per-feature Gaussian parameters.
        Args:
            X: (n_samples, n_features)  — training features
            y: (n_samples,)             — integer class labels
        """
        self.classes_ = np.unique(y)           # distinct class labels
        self.n_classes = len(self.classes_)
        n_features = X.shape[1]

        # Storage: mean, std, and log-prior per class
        self.means_  = np.zeros((self.n_classes, n_features))
        self.stds_   = np.zeros((self.n_classes, n_features))
        self.log_priors_ = np.zeros(self.n_classes)

        for i, cls in enumerate(self.classes_):
            X_cls = X[y == cls]          # all training samples of this class
            self.means_[i] = X_cls.mean(axis=0)           # μ per feature
            self.stds_[i]  = X_cls.std(axis=0) + 1e-9    # σ + tiny epsilon (avoid div/0)
            # Prior = log(fraction of training samples in this class)
            self.log_priors_[i] = np.log(len(X_cls) / len(X))

        return self

    def _log_gaussian(self, X, mu, sigma):
        """Log of Gaussian PDF: log N(x; mu, sigma) for all samples and features.
        Returns shape (n_samples, n_features) — sum across features for total log-likelihood."""
        return -0.5 * np.log(2 * np.pi * sigma**2) - ((X - mu)**2) / (2 * sigma**2)

    def predict_log_proba(self, X):
        """Return log-probability scores for each class. Shape (n_samples, n_classes)."""
        log_scores = np.zeros((len(X), self.n_classes))
        for i, cls in enumerate(self.classes_):
            # Log-prior + sum of log-likelihoods across all features (independence assumption!)
            log_likelihood = self._log_gaussian(X, self.means_[i], self.stds_[i])
            log_scores[:, i] = self.log_priors_[i] + log_likelihood.sum(axis=1)
        return log_scores

    def predict(self, X):
        """Return predicted class label (argmax of log-probability scores)."""
        log_scores = self.predict_log_proba(X)
        class_indices = np.argmax(log_scores, axis=1)
        return self.classes_[class_indices]

# ── Fit and evaluate the scratch implementation ────────────────────────────────
gnb_scratch = GaussianNBScratch()
gnb_scratch.fit(X_tr, y_tr)
y_pred_scratch = gnb_scratch.predict(X_te)
acc_scratch = accuracy_score(y_te, y_pred_scratch)

print('Learned parameters (our scratch implementation):')
print(f'  Classes: {gnb_scratch.classes_}')
for i, cls_name in enumerate(['Spam', 'Legitimate']):
    means = gnb_scratch.means_[i]
    print(f'  {cls_name} means: {dict(zip(feature_names, means.round(1)))}')

print(f'\nScratch GNB accuracy: {acc_scratch:.4f}')

In [ ]:
# ── Compare scratch vs sklearn GaussianNB ─────────────────────────────────────
t0 = time.perf_counter()
gnb_sklearn = GaussianNB()
gnb_sklearn.fit(X_tr, y_tr)
t_gnb = time.perf_counter() - t0

y_pred_gnb  = gnb_sklearn.predict(X_te)
acc_gnb     = accuracy_score(y_te, y_pred_gnb)
y_proba_gnb = gnb_sklearn.predict_proba(X_te)  # shape (n_test, 2)

print(f'sklearn GaussianNB accuracy : {acc_gnb:.4f}  |  Train time: {t_gnb:.6f}s')
print(f'Our scratch NB accuracy     : {acc_scratch:.4f}')
print(f'(Small differences may exist from variance estimator: sklearn uses var, we use std)')

# Check agreement between the two implementations
agreement = (gnb_sklearn.predict(X_te) == gnb_scratch.predict(X_te)).mean()
print(f'Agreement between implementations: {agreement:.2%}')

print(f'\nClassification Report (sklearn GaussianNB):')
print(classification_report(y_te, y_pred_gnb, target_names=['Spam', 'Legitimate']))

## Section 5: Multinomial Naïve Bayes

### When to Use Multinomial NB

Gaussian NB assumes features follow a bell curve — great for email_length or formal_words count, but not for raw word frequencies. **Multinomial NB** models features as word counts from a categorical distribution:

$$P(x_j \mid y = k) \propto \theta_{jk}^{x_j}$$

where $\theta_{jk}$ is the probability of word j appearing in a document of class k, estimated as:

$$\theta_{jk} = \frac{\text{count of word } j \text{ in class-} k \text{ emails}}{\text{total words in class-} k \text{ emails}}$$

**Laplace smoothing (add-1 smoothing):**
$$\theta_{jk} = \frac{\text{count}(j, k) + \alpha}{\text{total words in class } k + \alpha \cdot V}$$

where $\alpha = 1$ (Laplace) and $V$ = vocabulary size. This ensures no word gets zero probability.

In [ ]:
# ── Create bag-of-words feature matrix for 5 vocabulary words ─────────────────
np.random.seed(42)

# Vocabulary: 5 characteristic words
VOCAB = ['winner', 'free', 'dear', 'regards', 'meeting']
n_bow = 800  # total emails

# Word count distributions per class (Poisson means)
# Spam:   high 'winner'/'free', low 'dear'/'regards'/'meeting'
# Legit:  low 'winner'/'free',  high 'dear'/'regards'/'meeting'
spam_bow  = np.column_stack([
    np.random.poisson(3.0, n_bow//2),  # winner: common in spam
    np.random.poisson(2.5, n_bow//2),  # free:   common in spam
    np.random.poisson(0.2, n_bow//2),  # dear:   rare in spam
    np.random.poisson(0.1, n_bow//2),  # regards: rare in spam
    np.random.poisson(0.1, n_bow//2),  # meeting: very rare in spam
])
legit_bow = np.column_stack([
    np.random.poisson(0.1, n_bow//2),  # winner: very rare in legit
    np.random.poisson(0.3, n_bow//2),  # free:   rare in legit
    np.random.poisson(2.0, n_bow//2),  # dear:   common in legit
    np.random.poisson(1.5, n_bow//2),  # regards: common in legit
    np.random.poisson(1.2, n_bow//2),  # meeting: common in legit
])

X_bow = np.vstack([spam_bow, legit_bow]).astype(int)  # word counts must be integers
y_bow = np.array([0]*(n_bow//2) + [1]*(n_bow//2))   # 0=spam, 1=legitimate

df_bow = pd.DataFrame(X_bow, columns=VOCAB)
df_bow['label'] = ['Spam' if y == 0 else 'Legit' for y in y_bow]

print('Bag-of-Words matrix — first 5 rows per class:')
print(df_bow.groupby('label')[VOCAB].mean().round(2).to_string())
print(f'\nMatrix shape: {X_bow.shape} (800 emails × 5 vocabulary words)')

In [ ]:
# ── Train and evaluate MultinomialNB ──────────────────────────────────────────
X_bow_tr, X_bow_te, y_bow_tr, y_bow_te = train_test_split(
    X_bow, y_bow, test_size=0.25, random_state=42, stratify=y_bow
)

t0 = time.perf_counter()
mnb = MultinomialNB(alpha=1.0)   # alpha=1 is Laplace smoothing
mnb.fit(X_bow_tr, y_bow_tr)
t_mnb = time.perf_counter() - t0

y_pred_mnb = mnb.predict(X_bow_te)
acc_mnb    = accuracy_score(y_bow_te, y_pred_mnb)

print(f'MultinomialNB accuracy : {acc_mnb:.4f}  |  Train time: {t_mnb:.6f}s')
print(f'\nLearned log word probabilities per class:')
print(f'  (These are log θ_jk — exponentiate for actual probabilities)')
for i, cls_name in enumerate(['Spam', 'Legit']):
    log_probs = mnb.feature_log_prob_[i]
    probs = np.exp(log_probs)
    word_probs = {w: round(p, 4) for w, p in zip(VOCAB, probs)}
    print(f'  {cls_name}: {word_probs}')

# Predict on a new email: [winner=2, free=1, dear=0, regards=0, meeting=0]
new_email = np.array([[2, 1, 0, 0, 0]])
pred_class = mnb.predict(new_email)[0]
pred_proba = mnb.predict_proba(new_email)[0]
print(f'\nNew email: winner=2, free=1, dear=0, regards=0, meeting=0')
print(f'  P(spam)  = {pred_proba[0]:.4f}')
print(f'  P(legit) = {pred_proba[1]:.4f}')
print(f'  Prediction: {"Spam" if pred_class == 0 else "Legit"}')

## Section 6: Laplace Smoothing — The Zero-Probability Problem

### The Problem

Suppose the word `"xyzzy"` never appeared in any spam email during training. Then:
$$P(\text{xyzzy} \mid \text{spam}) = \frac{0}{\text{total spam words}} = 0$$

Now a test email contains `"xyzzy"`. The spam likelihood becomes:
$$P(\text{words} \mid \text{spam}) = \ldots \times P(\text{xyzzy} \mid \text{spam}) = \ldots \times 0 = 0$$

**One unseen word destroys the entire probability.** This is catastrophically bad.

### Laplace Smoothing (Add-1)

Pretend you saw every word **at least once** by adding a pseudo-count $\alpha$:

$$\theta_{jk} = \frac{\text{count}(j, k) + \alpha}{\sum_j \text{count}(j, k) + \alpha \cdot V}$$

With $\alpha = 1$, every word has a minimum probability of $\frac{1}{\text{total words} + V}$.

In [ ]:
# ── Laplace smoothing demonstration ───────────────────────────────────────────
print('=== Zero-Probability Problem & Laplace Smoothing ===')
print()

# Simulate: 'winner' appears in training; 'xyzzy' does NOT appear at all
# Training corpus: 'winner' appears 90 times in spam, 'xyzzy' appears 0 times
total_spam_words = 10_000
count_winner_spam = 90
count_xyzzy_spam  = 0      # unseen word
vocab_size = 6             # winner, free, dear, regards, meeting, xyzzy

print('Without smoothing (alpha=0):')
p_winner_no_smooth = count_winner_spam / total_spam_words
p_xyzzy_no_smooth  = count_xyzzy_spam  / total_spam_words
print(f'  P(winner | spam) = {count_winner_spam}/{total_spam_words} = {p_winner_no_smooth:.5f}')
print(f'  P(xyzzy  | spam) = {count_xyzzy_spam}/{total_spam_words}  = {p_xyzzy_no_smooth}  ← ZERO!')
print(f'  Joint score if email has (winner, xyzzy): {p_winner_no_smooth:.5f} × {p_xyzzy_no_smooth} = 0.0')
print(f'  Result: spam probability = 0.0 regardless of all other words — BROKEN.')

print()
print('With Laplace smoothing (alpha=1):')
for alpha in [0.001, 0.1, 1, 5, 10]:
    p_winner = (count_winner_spam + alpha) / (total_spam_words + alpha * vocab_size)
    p_xyzzy  = (count_xyzzy_spam  + alpha) / (total_spam_words + alpha * vocab_size)
    print(f'  alpha={alpha:5.3f}: P(winner|spam)={p_winner:.5f}, P(xyzzy|spam)={p_xyzzy:.6f}')

print()
print('Effect of alpha: larger alpha = more smoothing = probabilities pulled toward uniform.')
print('alpha=1 (Laplace) is a common default. alpha=0.1 (Lidstone) is lighter smoothing.')

In [ ]:
# ── Visualise: effect of alpha on estimated word probabilities ─────────────────
# Compare MultinomialNB trained with different alpha values
alphas = [1e-6, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
accs_by_alpha = []

for alpha in alphas:
    mnb_a = MultinomialNB(alpha=alpha)
    mnb_a.fit(X_bow_tr, y_bow_tr)
    acc_a = accuracy_score(y_bow_te, mnb_a.predict(X_bow_te))
    accs_by_alpha.append(acc_a)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Effect of Laplace Smoothing Parameter α on Multinomial NB', fontsize=12, fontweight='bold')

# Left: accuracy vs alpha
axes[0].semilogx(alphas, accs_by_alpha, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].axvline(1.0, color='red', linestyle='--', label='α=1 (Laplace)')
axes[0].set_xlabel('α (smoothing parameter, log scale)')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Accuracy vs Smoothing Strength')
axes[0].legend(); axes[0].set_ylim(0.7, 1.01)

# Right: how smoothing affects probability estimates for each word in spam class
alpha_plot = [0.001, 1.0, 10.0]
colors_plot = ['#e74c3c', '#2ecc71', '#3498db']
x_pos = np.arange(len(VOCAB))
bar_w = 0.25

for j, (a, col) in enumerate(zip(alpha_plot, colors_plot)):
    mnb_temp = MultinomialNB(alpha=a).fit(X_bow_tr, y_bow_tr)
    spam_probs = np.exp(mnb_temp.feature_log_prob_[0])  # spam class word probs
    axes[1].bar(x_pos + j*bar_w, spam_probs, bar_w, label=f'α={a}', color=col, alpha=0.85)

axes[1].set_xticks(x_pos + bar_w)
axes[1].set_xticklabels(VOCAB, rotation=20)
axes[1].set_ylabel('P(word | spam)')
axes[1].set_title('Word Probabilities in Spam Class\n(more smoothing → flatter distribution)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Section 7: NB as a Generative Model — Sampling Synthetic Emails

Unlike logistic regression (which only learns the decision boundary), Gaussian NB learns a **complete model of how each class generates data**. This means we can **sample** new synthetic emails from the learned distributions.

This is the generative model's superpower: it can create new data, not just classify it.

In [ ]:
# ── Generative sampling: create synthetic emails from learned Gaussian NB ──────
np.random.seed(99)

def sample_from_gaussian_nb(gnb_model, class_idx, n_samples, feature_names):
    """
    Sample synthetic data from the learned Gaussian distributions.
    For each feature j in class k, sample from N(mu_jk, sigma_jk).
    This exploits the generative nature of Naïve Bayes.
    """
    # sklearn GaussianNB stores: theta_ = means, var_ = variances
    means = gnb_model.theta_[class_idx]        # shape (n_features,)
    stds  = np.sqrt(gnb_model.var_[class_idx]) # convert variance to std

    samples = np.column_stack([
        np.random.normal(mu, sigma, n_samples).clip(0)   # clip: no negative counts
        for mu, sigma in zip(means, stds)
    ])
    return pd.DataFrame(samples, columns=feature_names)

# Sample 10 synthetic emails from each class
synthetic_spam  = sample_from_gaussian_nb(gnb_sklearn, 0, 10, feature_names)
synthetic_legit = sample_from_gaussian_nb(gnb_sklearn, 1, 10, feature_names)

print('=== 5 Synthetic Spam Emails (sampled from learned Gaussian NB) ===')
print(synthetic_spam.head().round(1).to_string())

print('\n=== 5 Synthetic Legitimate Emails ===')
print(synthetic_legit.head().round(1).to_string())

# Are the synthetic samples classified correctly by the same NB model?
pred_synth_spam  = gnb_sklearn.predict(synthetic_spam.values)
pred_synth_legit = gnb_sklearn.predict(synthetic_legit.values)
print(f'\nSynthetic spam classified as spam   : {(pred_synth_spam == 0).sum()}/10')
print(f'Synthetic legit classified as legit : {(pred_synth_legit == 1).sum()}/10')
print('(Not 100% because sampling adds random noise — expected behaviour)')

In [ ]:
# ── Visualise: real data vs generative samples per class ──────────────────────
n_gen = 300   # generate many samples to compare distributions visually
gen_spam  = sample_from_gaussian_nb(gnb_sklearn, 0, n_gen, feature_names)
gen_legit = sample_from_gaussian_nb(gnb_sklearn, 1, n_gen, feature_names)

# Compare: email_length and num_links for real vs generated data
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Real Data vs Generative NB Samples\n'
             '(good generative model = similar distribution)', fontsize=12, fontweight='bold')

for row, feat in enumerate(['email_length', 'num_links']):
    real_spam_vals  = df_bin[df_bin['label'] == 'Spam'][feat].values
    real_legit_vals = df_bin[df_bin['label'] == 'Legitimate'][feat].values

    # Left column: Spam
    axes[row, 0].hist(real_spam_vals,  bins=30, alpha=0.6, color=SPAM_COLOR,
                      label='Real spam',  density=True)
    axes[row, 0].hist(gen_spam[feat].values, bins=30, alpha=0.6, color='lightsalmon',
                      label='Generated spam', density=True)
    axes[row, 0].set_title(f'SPAM — {feat}', fontweight='bold')
    axes[row, 0].set_xlabel(feat); axes[row, 0].set_ylabel('Density')
    axes[row, 0].legend(fontsize=9)

    # Right column: Legitimate
    axes[row, 1].hist(real_legit_vals,       bins=30, alpha=0.6, color=LEGIT_COLOR,
                      label='Real legit',     density=True)
    axes[row, 1].hist(gen_legit[feat].values, bins=30, alpha=0.6, color='lightblue',
                      label='Generated legit', density=True)
    axes[row, 1].set_title(f'LEGIT — {feat}', fontweight='bold')
    axes[row, 1].set_xlabel(feat); axes[row, 1].set_ylabel('Density')
    axes[row, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Generated samples should follow the same shape as real data.')
print('This confirms NB correctly captured the class-conditional distributions.')

## Section 8: Comparing NB Variants

Three Naïve Bayes variants exist in sklearn, each for different feature types:

| Variant | Feature Type | Likelihood Model | Use Case |
|---------|-------------|------------------|----------|
| **GaussianNB** | Continuous (real-valued) | $P(x_j\|y=k) = \mathcal{N}(\mu_{jk}, \sigma^2_{jk})$ | Measurements, email feature counts |
| **MultinomialNB** | Non-negative integers (counts) | $P(x_j\|y=k) \propto \theta_{jk}^{x_j}$ | Word frequencies, TF-IDF |
| **BernoulliNB** | Binary (0/1) | $P(x_j\|y=k) = p_{jk}^{x_j}(1-p_{jk})^{1-x_j}$ | Word presence/absence |

In [ ]:
# ── Compare all three NB variants on the bag-of-words data ────────────────────
# Binarise word counts for BernoulliNB (1 = word appears, 0 = does not)
X_bow_binary = (X_bow > 0).astype(int)   # presence/absence
X_bow_bin_tr, X_bow_bin_te, _, _ = train_test_split(
    X_bow_binary, y_bow, test_size=0.25, random_state=42, stratify=y_bow
)

models = [
    ('GaussianNB',    GaussianNB(),        X_bow_tr, X_bow_te),
    ('MultinomialNB', MultinomialNB(alpha=1.0), X_bow_tr, X_bow_te),
    ('BernoulliNB',   BernoulliNB(alpha=1.0),   X_bow_bin_tr, X_bow_bin_te),
]

print('=== NB Variants Comparison on Bag-of-Words Email Data ===')
print(f'{"Model":<18} {"Accuracy":>10} {"Train Time (s)":>16}')
print('-' * 46)

for name, model, X_tr_v, X_te_v in models:
    t0 = time.perf_counter()
    model.fit(X_tr_v, y_bow_tr)
    t = time.perf_counter() - t0
    acc = accuracy_score(y_bow_te, model.predict(X_te_v))
    print(f'{name:<18} {acc:>10.4f} {t:>16.6f}')

print()
print('GaussianNB assumes continuous — works but Multinomial is better for counts.')
print('MultinomialNB is the gold standard for word-count bag-of-words features.')
print('BernoulliNB ignores HOW MANY times a word appears — only presence matters.')

## Section 9: NB vs Logistic Regression — Head-to-Head

Naïve Bayes and Logistic Regression are both used for text classification. They have a theoretically interesting relationship:
- **NB** is generative: learns P(x|y) and uses Bayes' theorem
- **LR** is discriminative: directly learns P(y|x)

Ng & Jordan (2002) showed that NB converges faster with less data, but LR achieves better **asymptotic** accuracy with enough data.

In [ ]:
# ── NB vs Logistic Regression: accuracy, speed, and calibration ───────────────
# Use the continuous features (X_tr / X_te) for a fair comparison
scaler_nb = StandardScaler()
X_tr_sc = scaler_nb.fit_transform(X_tr)
X_te_sc = scaler_nb.transform(X_te)

competitors = [
    ('GaussianNB',   GaussianNB(),          X_tr,    X_te),
    ('LogisticReg',  LogisticRegression(max_iter=500, random_state=42), X_tr_sc, X_te_sc),
]

print('=== Gaussian NB vs Logistic Regression ===')
print(f'{"Model":<18} {"Accuracy":>10} {"Train Time (s)":>16} {"Calibration (ECE)":>20}')
print('-' * 66)

def expected_calibration_error(y_true, y_proba, n_bins=10):
    """Compute Expected Calibration Error (lower = better calibrated).
    Divides samples into probability bins; checks if stated confidence
    matches actual accuracy within each bin."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_proba >= bins[i]) & (y_proba < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_acc  = y_true[mask].mean()       # actual accuracy in bin
        bin_conf = y_proba[mask].mean()      # mean predicted confidence
        ece += mask.sum() * abs(bin_acc - bin_conf) / len(y_true)
    return ece

for name, model, X_tr_v, X_te_v in competitors:
    t0 = time.perf_counter()
    model.fit(X_tr_v, y_tr)
    t = time.perf_counter() - t0
    acc = accuracy_score(y_te, model.predict(X_te_v))
    proba_spam = model.predict_proba(X_te_v)[:, 1]  # P(spam)
    ece = expected_calibration_error(y_te, proba_spam)
    print(f'{name:<18} {acc:>10.4f} {t:>16.6f} {ece:>20.4f}')

print()
print('ECE = Expected Calibration Error: lower is better calibrated.')
print('LR typically beats NB on accuracy with enough data.')
print('NB trains near-instantly even on millions of documents.')

In [ ]:
# ── Learning curve: NB vs LR as training set size grows ───────────────────────
# Ng & Jordan (2002): NB converges faster with small data; LR wins asymptotically
train_fractions = np.linspace(0.05, 0.85, 15)
accs_nb_lc = []
accs_lr_lc = []

for frac in train_fractions:
    X_sub, _, y_sub, _ = train_test_split(X_tr, y_tr, train_size=frac,
                                           random_state=42, stratify=y_tr)
    X_sub_sc = scaler_nb.transform(X_sub)  # use already-fitted scaler

    # GaussianNB
    gnb_lc = GaussianNB().fit(X_sub, y_sub)
    accs_nb_lc.append(accuracy_score(y_te, gnb_lc.predict(X_te)))

    # Logistic Regression
    lr_lc = LogisticRegression(max_iter=500, random_state=42).fit(X_sub_sc, y_sub)
    accs_lr_lc.append(accuracy_score(y_te, lr_lc.predict(X_te_sc)))

n_train_samples = (train_fractions * len(X_tr)).astype(int)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_train_samples, accs_nb_lc, 'o-', color=SPAM_COLOR,  linewidth=2.5,
        markersize=7, label='Gaussian Naïve Bayes')
ax.plot(n_train_samples, accs_lr_lc, 's-', color=LEGIT_COLOR, linewidth=2.5,
        markersize=7, label='Logistic Regression')
ax.set_xlabel('Number of Training Samples', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Learning Curve: NB vs Logistic Regression\n'
             '(NB may lead with very small datasets; LR often catches up)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0.5, 1.02)
plt.tight_layout()
plt.show()

## Section 10: Log-Space Computation — Avoiding Underflow

Multiplying hundreds of small probabilities causes **floating-point underflow** to zero.

An email with 200 words, each with probability ~0.01, gives:
$$0.01^{200} = 10^{-400} \approx 0$$

This rounds to zero in 64-bit floating point (minimum ~$10^{-308}$). The fix: work in log-space.

In [ ]:
# ── Log-space computation demo ─────────────────────────────────────────────────
print('=== Log-Space Computation: Why It Matters ===')
print()

# Simulate an email with 200 words, each with individual probability 0.01
n_words = 200
individual_prob = 0.01

# Naive: multiply probabilities directly
naive_product = individual_prob ** n_words
print(f'Naive product: {individual_prob}^{n_words} = {naive_product}')
print(f'  → Rounds to ZERO (underflow at ~10^-308)!')

# Log-space: sum logarithms
log_sum = n_words * np.log(individual_prob)
print(f'\nLog-space: {n_words} × log({individual_prob}) = {log_sum:.2f}')
print(f'  → This number is perfectly representable in float64.')
print(f'  → For classification, we compare log-scores — no need to exponentiate.')

print()
print('Log-space Naïve Bayes prediction rule:')
print('  ŷ = argmax_k [ log P(y=k) + Σⱼ log P(xⱼ | y=k) ]')
print()

# Demonstrate: compare classes in log-space
log_spam_score  = np.log(0.40) + n_words * np.log(0.015)  # 40% prior, avg 1.5% word prob
log_legit_score = np.log(0.60) + n_words * np.log(0.008)  # 60% prior, avg 0.8% word prob

print(f'Example log-scores for a spam-like email ({n_words} words):')
print(f'  log P(spam)  + Σ log P(words|spam)  = {log_spam_score:.2f}')
print(f'  log P(legit) + Σ log P(words|legit) = {log_legit_score:.2f}')
print(f'  Predicted class: {"SPAM" if log_spam_score > log_legit_score else "LEGITIMATE"}')
print(f'  (Higher log-score wins — correct even without converting back to probabilities)')

## Self-Check Questions

Try to answer each question before expanding the solution.

---

**Question 1:** Words "cheap" and "pills" co-occur in spam emails. This means they are **correlated** in reality. Does this violate the Naïve Bayes independence assumption? If yes, does NB still work?

<details>
<summary>Click to reveal answer</summary>

**Yes, it violates the independence assumption.** Naïve Bayes assumes:
$$P(\text{cheap}, \text{pills} \mid \text{spam}) = P(\text{cheap} \mid \text{spam}) \times P(\text{pills} \mid \text{spam})$$

But if "cheap pills" is a spam phrase where both words nearly always appear together, then seeing "cheap" already tells you a lot about whether "pills" will appear — they are not independent.

**Despite this violation, NB often still works well for two reasons:**

1. **Decision correctness vs probability correctness:** Even if the individual probabilities are wrong, the *ranking* (which class gets the highest score) is often preserved. NB can predict the correct class even when its stated probability values are miscalibrated.

2. **Biased but consistent:** The correlated features effectively get "double-counted" — seeing "cheap" and seeing "pills" both push toward spam, as if they were independent strong evidence. In practice this bias often amplifies rather than reverses the correct decision.

This is why NB is described as "the model that shouldn't work but does."

</details>

---

**Question 2:** A test email contains the word `"xyzzy"` that never appeared in training data. Without Laplace smoothing, what is P("xyzzy" | spam)? Why is this a problem?

<details>
<summary>Click to reveal answer</summary>

Without Laplace smoothing:

$$P(\text{xyzzy} \mid \text{spam}) = \frac{\text{count("xyzzy" in spam training emails)}}{\text{total words in spam training emails}} = \frac{0}{\text{something}} = 0$$

**Why this is catastrophic:** The Naïve Bayes spam score is a *product* of all word likelihoods:
$$P(\text{all words} \mid \text{spam}) = P(w_1 \mid \text{spam}) \times \ldots \times P(\text{xyzzy} \mid \text{spam}) = \ldots \times 0 = 0$$

One unseen word makes the entire spam probability zero. This means the model would classify any email containing even one new word as non-spam with certainty — even if 99 other words are extremely spam-like.

**Laplace smoothing** (add $\alpha = 1$ to every word count) ensures:
$$P(\text{xyzzy} \mid \text{spam}) = \frac{0 + 1}{\text{total words} + V} > 0$$

Every unseen word now has a small but non-zero probability, preserving the product.

</details>

---

**Question 3:** Gaussian NB is **generative** (models P(x|y)) while Logistic Regression is **discriminative** (models P(y|x) directly). Despite often having lower accuracy, when would you **prefer** the generative model?

<details>
<summary>Click to reveal answer</summary>

There are several practical scenarios where Gaussian NB (generative) is preferred over Logistic Regression (discriminative):

1. **Very small training data:** NB converges to its best performance with far fewer samples than LR. Ng & Jordan (2002) showed that NB reaches near-optimal accuracy at O(log n) samples, while LR needs O(n). If you have 50 emails, NB may outperform LR.

2. **Anomaly detection:** Because NB models P(x|y), you can compute how likely a new email is *under the spam model*. Emails with very low P(x|spam) AND low P(x|legit) are anomalous — LR cannot detect these without additional tricks.

3. **Data generation / augmentation:** NB can synthesise new training examples by sampling from P(x|y), as we demonstrated. LR cannot generate new data.

4. **Missing features at test time:** If some features are missing for a test email, NB can simply omit those likelihood terms (the product remains valid). LR with missing features requires imputation or special handling.

5. **Interpretability:** The learned Gaussians (μ, σ per class per feature) are highly interpretable. You can directly see *what a spam email looks like* in feature space.

**Rule of thumb:** Use NB when data is scarce, interpretability matters, or you need generative capabilities. Use LR when you have enough data and classification accuracy is the primary goal.

</details>

## Summary

| Concept | Key Idea |
|---------|----------|
| **Bayes' Theorem** | P(class\|features) ∝ P(features\|class) × P(class) |
| **Prior** | P(spam) — base rate before seeing the email |
| **Likelihood** | P(words\|spam) — how probable are these features in spam |
| **Posterior** | P(spam\|words) — updated belief after seeing the email |
| **Naive assumption** | Features are independent given class — wrong but effective |
| **Gaussian NB** | Models each continuous feature as a Gaussian per class |
| **Multinomial NB** | Models word counts; best for bag-of-words text classification |
| **Bernoulli NB** | Models binary word presence/absence |
| **Laplace smoothing** | Add α to counts to prevent zero-probability for unseen words |
| **Log-space** | Sum logs instead of multiply probabilities — avoids underflow |
| **Generative model** | NB learns P(x\|y) → can synthesise new data; LR cannot |
| **NB vs LR** | NB faster/better with small data; LR more accurate with large data |

**Week 6 complete.** You now know how to:
- Frame binary and multi-class classification with logistic regression and softmax
- Extend to OvR, OvO, and joint multinomial training
- Apply a probabilistic generative framework via Naïve Bayes
- Handle unseen words with Laplace smoothing
- Choose between discriminative and generative approaches based on data size and task